# 26. GPT-4o pseudo-label quality analysis

Comprehensive analysis of GPT-4o zero-shot pseudo-label quality on the HumAID dataset,
covering per-class performance, error patterns, confidence calibration, and prompt sensitivity.

## Sections

1. **Per-class quality** - Precision, recall, F1 per class across all 10 events
2. **Error analysis** - Confusion matrix and top misclassification patterns
3. **Confidence calibration** - Accuracy at each GPT-4o confidence level
4. **Prompt sensitivity** - Comparing three prompt variants on the dev split

## Setup

In [ ]:
import csv
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Cannot find repo root: no ancestor directory contains '{marker}/'")

repo_root = _find_repo_root()
print(f"Repo root: {repo_root}")

PSEUDO_LABEL_DIR = repo_root / "data" / "pseudo-labelled" / "gpt-4o"
EVENTS = [
    "california_wildfires_2018", "canada_wildfires_2016", "cyclone_idai_2019",
    "hurricane_dorian_2019", "hurricane_florence_2018", "hurricane_harvey_2017",
    "hurricane_irma_2017", "hurricane_maria_2017", "kaikoura_earthquake_2016",
    "kerala_floods_2018",
]

# Load all pseudo-labeled data
all_rows = []
for event in EVENTS:
    csv_path = PSEUDO_LABEL_DIR / event / f"{event}_train_pred.csv"
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found")
        continue
    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row["event"] = event
            row["confidence"] = float(row["confidence"])
            all_rows.append(row)

print(f"Total pseudo-labeled tweets: {len(all_rows):,}")
print(f"Events loaded: {len(set(r['event'] for r in all_rows))}")

## 1. Per-class pseudo-label quality

Precision, recall, and F1 for each class, pooled across all 10 events.

In [ ]:
classes = sorted(set(r["class_label"] for r in all_rows))

# Compute per-class TP, FP, FN
tp = Counter()
fp = Counter()
fn = Counter()

for r in all_rows:
    true = r["class_label"]
    pred = r["predicted_label"]
    if true == pred:
        tp[true] += 1
    else:
        fn[true] += 1
        fp[pred] += 1

print("=" * 90)
print("PER-CLASS GPT-4o PSEUDO-LABEL QUALITY (pooled across 10 events)")
print("=" * 90)
print(f'{"Class":<45s} {"Support":>8s} {"P":>8s} {"R":>8s} {"F1":>8s}')
print("-" * 90)

macro_f1_vals = []
for cls in classes:
    support = tp[cls] + fn[cls]
    p = tp[cls] / (tp[cls] + fp[cls]) if (tp[cls] + fp[cls]) > 0 else 0
    r = tp[cls] / (tp[cls] + fn[cls]) if (tp[cls] + fn[cls]) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    macro_f1_vals.append(f1)
    print(f"{cls:<45s} {support:>8d} {p:>8.3f} {r:>8.3f} {f1:>8.3f}")

total = len(all_rows)
correct = sum(1 for r in all_rows if r["class_label"] == r["predicted_label"])
macro_f1 = sum(macro_f1_vals) / len(macro_f1_vals)
print("-" * 90)
print(f'{"Overall":<45s} {total:>8d} {"":>8s} {"":>8s} {macro_f1:>8.3f}')
print(f"Accuracy: {correct}/{total} = {correct/total:.3f}")

## 2. Error analysis

Confusion matrix and top misclassification patterns.

In [ ]:
# Full confusion matrix
confusion = defaultdict(Counter)
for r in all_rows:
    confusion[r["class_label"]][r["predicted_label"]] += 1

print("=" * 100)
print("TOP MISCLASSIFICATION PATTERNS (true -> predicted, count, % of true class)")
print("=" * 100)

# Collect all (true, pred, count) where true != pred
errors = []
for true_cls in classes:
    total_true = sum(confusion[true_cls].values())
    for pred_cls in classes:
        if true_cls != pred_cls:
            count = confusion[true_cls][pred_cls]
            if count > 0:
                pct = count / total_true * 100
                errors.append((true_cls, pred_cls, count, pct))

errors.sort(key=lambda x: -x[2])
print(f'{"True class":<40s} {"Predicted as":<40s} {"Count":>6s} {"% of true":>9s}')
print("-" * 100)
for true_cls, pred_cls, count, pct in errors[:20]:
    print(f"{true_cls:<40s} {pred_cls:<40s} {count:>6d} {pct:>8.1f}%")

In [ ]:
# Detailed breakdown: "Other relevant information" errors
target_class = "other_relevant_information"
total_target = sum(confusion[target_class].values())

print("=" * 80)
print(f'BREAKDOWN: GPT-4o predictions on "{target_class}" tweets (n={total_target:,})')
print("=" * 80)
print(f'{"GPT-4o predicted label":<45s} {"Count":>6s} {"% of total":>10s}')
print("-" * 65)

for pred_cls, count in confusion[target_class].most_common():
    pct = count / total_target * 100
    marker = " (correct)" if pred_cls == target_class else ""
    print(f"{pred_cls + marker:<45s} {count:>6d} {pct:>9.1f}%")

## 3. Confidence calibration

Accuracy at each GPT-4o self-reported confidence level.

In [ ]:
# Group by confidence level
by_conf = defaultdict(lambda: {"total": 0, "correct": 0})

for r in all_rows:
    conf = r["confidence"]
    by_conf[conf]["total"] += 1
    if r["class_label"] == r["predicted_label"]:
        by_conf[conf]["correct"] += 1

total_all = len(all_rows)

print("=" * 80)
print("GPT-4o CONFIDENCE CALIBRATION")
print("=" * 80)
print(f'{"Confidence":>10s} {"Count":>8s} {"% of total":>10s} {"Accuracy":>10s} {"Gap (conf-acc)":>14s}')
print("-" * 55)

for conf in sorted(by_conf.keys()):
    d = by_conf[conf]
    pct = d["total"] / total_all * 100
    acc = d["correct"] / d["total"] if d["total"] > 0 else 0
    gap = conf - acc
    print(f"{conf:>10.2f} {d['total']:>8d} {pct:>9.1f}% {acc:>10.3f} {gap:>+13.3f}")

print()
print("Key observations:")
if 0.95 in by_conf:
    d95 = by_conf[0.95]
    acc95 = d95["correct"] / d95["total"]
    print(f"  - 0.95 confidence: {d95['total']:,} predictions ({d95['total']/total_all*100:.1f}%), accuracy = {acc95:.3f}")
if 0.85 in by_conf:
    d85 = by_conf[0.85]
    acc85 = d85["correct"] / d85["total"]
    print(f"  - 0.85 confidence: {d85['total']:,} predictions ({d85['total']/total_all*100:.1f}%), accuracy = {acc85:.3f}")

## 4. Prompt sensitivity

Comparison of three prompt variants on the dev split.

- **RULES_1**: Compact single-line class definitions
- **RULES_2**: Medium detail with "PRIMARY INTENT" framing
- **RULES_3**: Detailed multi-line Definition/Include/Exclude criteria per class

Results loaded from the zeroshot analysis summaries.

In [ ]:
# Load prompt sensitivity results from zeroshot runs
import re

zeroshot_root = repo_root / "zeroshot"
prompt_results = defaultdict(dict)

# Search: zeroshot/results/{event}/dev/gpt-4o/{batch_id}/analysis/charts/summary.json
for summary_path in sorted(zeroshot_root.glob("results/*/dev/gpt-4o/*/analysis/charts/summary.json")):
    parts = summary_path.parts
    # Extract event from path
    event_idx = list(parts).index("results") + 1
    event = parts[event_idx]

    # Extract rule from batch directory name (e.g., "20260406-004223-modeS-gpt-4o-RULES1-filtered")
    batch_dir = parts[event_idx + 3]  # the batch ID directory
    rule_match = re.search(r"(RULES\d+)", batch_dir)
    if not rule_match:
        continue
    rule = rule_match.group(1)

    try:
        data = json.loads(summary_path.read_text(encoding="utf-8"))
        macro_f1 = data.get("macro_f1")
        if macro_f1 is not None:
            prompt_results[event][rule] = macro_f1
    except Exception:
        continue

if prompt_results:
    print("=" * 90)
    print("PROMPT SENSITIVITY: GPT-4o Macro-F1 on dev split")
    print("=" * 90)

    all_rules = sorted(set(r for ev in prompt_results.values() for r in ev.keys()))
    header = f'{"Event":<35s}'
    for rule in all_rules:
        header += f'  {rule:>10s}'
    print(header)
    print("-" * 90)

    rule_means = defaultdict(list)
    for event in EVENTS:
        if event not in prompt_results:
            continue
        row = f"{event:<35s}"
        vals = prompt_results[event]
        best_val = max(vals.values()) if vals else 0
        for rule in all_rules:
            v = vals.get(rule)
            if v is not None:
                rule_means[rule].append(v)
                marker = " *" if abs(v - best_val) < 0.0005 else "  "
                row += f"  {v:>8.3f}{marker}"
            else:
                row += f"       ---  "
        print(row)

    print("-" * 90)
    row = f'{"Mean":<35s}'
    for rule in all_rules:
        vals = rule_means[rule]
        if vals:
            row += f"  {sum(vals)/len(vals):>8.3f}  "
        else:
            row += f"       ---  "
    print(row)

    print()
    print(f"Events loaded: {len(prompt_results)}")
    print(f"Rules found: {all_rules}")
else:
    print("WARNING: No zeroshot summary files found.")
    print("Expected path: zeroshot/results/{event}/dev/gpt-4o/{batch}/analysis/charts/summary.json")

## Summary

This notebook provides a comprehensive analysis of GPT-4o pseudo-label quality:

1. **Per-class quality** varies substantially: F1 ranges from 0.276 (Other relevant information) to 0.885 (Injured or dead)
2. **Error patterns** show GPT-4o favors specific labels over vague ones, with Other relevant information frequently misclassified as Not humanitarian or Infrastructure damage
3. **Confidence calibration** reveals overconfidence: 49% of predictions get 0.95 confidence but only 79% are correct
4. **Prompt sensitivity** is low: three variants achieve Macro-F1 in the 0.601-0.613 range, with the simplest prompt performing best